In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q torch_geometric

In [ ]:
import os, sys
BASE = "/content/drive/MyDrive/cicidsposter"
for f in ("CICDS2017.csv", "AttackGraph.txt", "models.py"):
    assert os.path.exists(os.path.join(BASE, f)), f"not found: {os.path.join(BASE, f)}"
sys.path.append(BASE)
os.chdir(BASE)
print("working folder:", os.getcwd())

working folder: /content/drive/MyDrive/cicidsposter


### Libraries

In [ ]:
print("Hello world")

Hello world


In [ ]:
import re
import pandas as pd
import numpy as np
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.nn import GCNConv
from models import GCN,NN
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cpu


### Attack Graph

In [ ]:
path = os.path.join(BASE, "AttackGraph.txt")

In [ ]:
with open(path, 'r') as f:
    content = f.read()

In [ ]:
print(content[:500])

1,"canAccessHost(machine3)","OR",0
2,"RULE 8 (Access a host through executing code on the machine)","AND",0
3,"execCode(machine3,user)","OR",0
4,"RULE 0 (When a principal is compromised any machine he has an account on will also be compromised)","AND",0
4,1,-1
5,"hasAccount(victim,machine3,user)","LEAF",1
4,5,-1
6,"principalCompromised(victim)","OR",0
7,"RULE 11 (password sniffing)","AND",0
8,"execCode(machine1,root)","OR",0
9,"RULE 2 (remote exploit of a server program)","AND",0
10,"netAccess(m


In [ ]:
edges = re.findall(r'^(\d+),(\d+),-1$', content, flags=re.MULTILINE)

In [ ]:
edges

[('4', '1'),
 ('4', '5'),
 ('11', '12'),
 ('11', '13'),
 ('10', '11'),
 ('9', '10'),
 ('9', '14'),
 ('9', '15'),
 ('8', '9'),
 ('7', '8'),
 ('7', '16'),
 ('6', '7'),
 ('19', '20'),
 ('19', '18'),
 ('18', '19'),
 ('21', '20'),
 ('25', '18'),
 ('24', '25'),
 ('26', '22'),
 ('24', '26'),
 ('29', '30'),
 ('29', '8'),
 ('28', '29'),
 ('31', '30'),
 ('35', '8'),
 ('34', '35'),
 ('36', '32'),
 ('34', '36'),
 ('33', '34'),
 ('33', '16'),
 ('33', '6'),
 ('32', '33'),
 ('31', '32'),
 ('28', '31'),
 ('27', '28'),
 ('38', '39'),
 ('37', '38'),
 ('27', '37'),
 ('24', '27'),
 ('23', '24'),
 ('23', '40'),
 ('23', '6'),
 ('22', '23'),
 ('21', '22'),
 ('18', '21'),
 ('17', '18'),
 ('17', '40'),
 ('6', '17'),
 ('41', '32'),
 ('41', '16'),
 ('6', '41'),
 ('42', '22'),
 ('42', '40'),
 ('6', '42'),
 ('43', '3'),
 ('43', '5'),
 ('6', '43'),
 ('4', '6'),
 ('3', '4'),
 ('2', '3'),
 ('1', '2'),
 ('46', '47'),
 ('46', '8'),
 ('45', '46'),
 ('48', '47'),
 ('48', '32'),
 ('45', '48'),
 ('44', '45'),
 ('50', '51')

In [ ]:
len(edges)

72

In [ ]:
raw_nodes = re.findall(r'^(\d+),"([^"]+)",(?:"?([^",\s]+)"?),(\d+)$', content, flags=re.MULTILINE)

In [ ]:
raw_nodes

[('1', 'canAccessHost(machine3)', 'OR', '0'),
 ('2',
  'RULE 8 (Access a host through executing code on the machine)',
  'AND',
  '0'),
 ('3', 'execCode(machine3,user)', 'OR', '0'),
 ('4',
  'RULE 0 (When a principal is compromised any machine he has an account on will also be compromised)',
  'AND',
  '0'),
 ('5', 'hasAccount(victim,machine3,user)', 'LEAF', '1'),
 ('6', 'principalCompromised(victim)', 'OR', '0'),
 ('7', 'RULE 11 (password sniffing)', 'AND', '0'),
 ('8', 'execCode(machine1,root)', 'OR', '0'),
 ('9', 'RULE 2 (remote exploit of a server program)', 'AND', '0'),
 ('10', 'netAccess(machine1,tcp,445)', 'OR', '0'),
 ('11', 'RULE 6 (direct network access)', 'AND', '0'),
 ('12', 'hacl(internet,machine1,tcp,445)', 'LEAF', '1'),
 ('13', 'attackerLocated(internet)', 'LEAF', '1'),
 ('14', 'networkServiceInfo(machine1,smbd,tcp,445,root)', 'LEAF', '1'),
 ('15',
  "vulExists(machine1,'CVE-2026-4408',smbd,remoteExploit,privEscalation)",
  'LEAF',
  '1'),
 ('16', 'hasAccount(victim,mach

In [ ]:
node_properties = []

In [ ]:
for item in raw_nodes:
    node_id = item[0]
    desc = item[1]
    node_type = item[2]
    val = item[3]

    if node_type == 'OR':
        shape = 'diamond'
    elif node_type == 'AND':
        shape = 'ellipse'
    else:
        shape = 'box'

    formatted_string = f'label="{node_id}:{desc}:{val}",shape={shape}'
    node_properties.append(formatted_string)

In [ ]:
node_properties

['label="1:canAccessHost(machine3):0",shape=diamond',
 'label="2:RULE 8 (Access a host through executing code on the machine):0",shape=ellipse',
 'label="3:execCode(machine3,user):0",shape=diamond',
 'label="4:RULE 0 (When a principal is compromised any machine he has an account on will also be compromised):0",shape=ellipse',
 'label="5:hasAccount(victim,machine3,user):1",shape=box',
 'label="6:principalCompromised(victim):0",shape=diamond',
 'label="7:RULE 11 (password sniffing):0",shape=ellipse',
 'label="8:execCode(machine1,root):0",shape=diamond',
 'label="9:RULE 2 (remote exploit of a server program):0",shape=ellipse',
 'label="10:netAccess(machine1,tcp,445):0",shape=diamond',
 'label="11:RULE 6 (direct network access):0",shape=ellipse',
 'label="12:hacl(internet,machine1,tcp,445):1",shape=box',
 'label="13:attackerLocated(internet):1",shape=box',
 'label="14:networkServiceInfo(machine1,smbd,tcp,445,root):1",shape=box',
 'label="15:vulExists(machine1,\'CVE-2026-4408\',smbd,remoteE

In [ ]:
len(node_properties)

51

In [ ]:
node_dict={}
nodes = [item[0] for item in raw_nodes]

In [ ]:
nodes

['1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 '10',
 '11',
 '12',
 '13',
 '14',
 '15',
 '16',
 '17',
 '18',
 '19',
 '20',
 '21',
 '22',
 '23',
 '24',
 '25',
 '26',
 '27',
 '28',
 '29',
 '30',
 '31',
 '32',
 '33',
 '34',
 '35',
 '36',
 '37',
 '38',
 '39',
 '40',
 '41',
 '42',
 '43',
 '44',
 '45',
 '46',
 '47',
 '48',
 '49',
 '50',
 '51']

In [ ]:
for item in node_properties:
    label_match=re.search(r'label="(.*)"',item)
    label_text=label_match.group(1)

    parts=label_text.split(':')
    node_id=nodes.index(parts[0])
    statement=parts[1]

    match=re.match(r'(.+)\((.*)\)',statement)
    predicate=match.group(1).strip()
    attr_text=match.group(2).strip()

    if ',' in attr_text:
        attributes=attr_text.split(',')
    else:
        attributes=attr_text.split()

    clean_attributes=[]
    for a in attributes:
        clean_attributes.append(a.strip("'\""))
    attributes=clean_attributes

    shape_match=re.search(r'shape=(\w+)',item)
    shape=shape_match.group(1)

    node_dict[node_id]={
        'predicate':predicate,
        'attributes':attributes,
        'shape':shape
    }

In [ ]:
len(node_dict)

51

In [ ]:
node_dict[0]

{'predicate': 'canAccessHost', 'attributes': ['machine3'], 'shape': 'diamond'}

### Encoding

In [ ]:
word2idx={}
idx=0

In [ ]:
for n in node_dict:
    node=node_dict[n]
    words=[node['predicate']]+node['attributes']

    for word in words:
        if word not in word2idx:
            word2idx[word]=idx
            idx+=1

In [ ]:
len(word2idx)

73

In [ ]:
word2idx

{'canAccessHost': 0,
 'machine3': 1,
 'RULE 8': 2,
 'Access': 3,
 'a': 4,
 'host': 5,
 'through': 6,
 'executing': 7,
 'code': 8,
 'on': 9,
 'the': 10,
 'machine': 11,
 'execCode': 12,
 'user': 13,
 'RULE 0': 14,
 'When': 15,
 'principal': 16,
 'is': 17,
 'compromised': 18,
 'any': 19,
 'he': 20,
 'has': 21,
 'an': 22,
 'account': 23,
 'will': 24,
 'also': 25,
 'be': 26,
 'hasAccount': 27,
 'victim': 28,
 'principalCompromised': 29,
 'RULE 11': 30,
 'password': 31,
 'sniffing': 32,
 'machine1': 33,
 'root': 34,
 'RULE 2': 35,
 'remote': 36,
 'exploit': 37,
 'of': 38,
 'server': 39,
 'program': 40,
 'netAccess': 41,
 'tcp': 42,
 '445': 43,
 'RULE 6': 44,
 'direct': 45,
 'network': 46,
 'access': 47,
 'hacl': 48,
 'internet': 49,
 'attackerLocated': 50,
 'networkServiceInfo': 51,
 'smbd': 52,
 'vulExists': 53,
 'CVE-2026-4408': 54,
 'remoteExploit': 55,
 'privEscalation': 56,
 'machine2': 57,
 'RULE 1': 58,
 'local': 59,
 'CVE-2021-3156': 60,
 'sudo': 61,
 'localExploit': 62,
 'RULE 9': 

In [ ]:
num_nodes=len(node_dict)
vocab_size=len(word2idx)

In [ ]:
node_features=np.zeros((num_nodes,vocab_size))

In [ ]:
node_features

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(51, 73))

In [ ]:
for node_id in node_dict:
    node=node_dict[node_id]
    words=[node['predicate']]+node['attributes']

    for word in words:
        col=word2idx[word]
        node_features[node_id][col]=1

In [ ]:
node_features.shape

(51, 73)

In [ ]:
node_features

array([[1., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       ...,
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 1., 0., ..., 0., 1., 0.]], shape=(51, 73))

### Extracting Privilege Nodes


In [ ]:
action_node_idx=[]

In [ ]:
for node_id in node_dict:
    node=node_dict[node_id]
    if node['shape']=='diamond':
        action_node_idx.append(node_id)

In [ ]:
len(action_node_idx)

14

In [ ]:
action_node_idx

[0, 2, 5, 7, 9, 17, 21, 23, 27, 31, 33, 36, 44, 48]

In [ ]:
adj_matrix=np.zeros((num_nodes,num_nodes))

In [ ]:
for edge in edges:
    child, parent = edge

    source_idx = nodes.index(parent)
    target_idx = nodes.index(child)

    adj_matrix[source_idx][target_idx] = 1

In [ ]:
edge_index=np.array(np.nonzero(adj_matrix))

In [ ]:
adj_matrix.shape

(51, 51)

In [ ]:
edge_index.shape

(2, 72)

### Dataset


In [ ]:
data = pd.read_csv(
    os.path.join(BASE, "CICDS2017.csv")
)

In [ ]:
data.shape

(2830743, 79)

In [ ]:
data.head

<bound method NDFrame.head of           Destination Port   Flow Duration   Total Fwd Packets  \
0                    54865               3                   2   
1                    55054             109                   1   
2                    55055              52                   1   
3                    46236              34                   1   
4                    54863               3                   2   
...                    ...             ...                 ...   
2830738                 53           32215                   4   
2830739                 53             324                   2   
2830740              58030              82                   2   
2830741                 53         1048635                   6   
2830742                 53           94939                   4   

          Total Backward Packets  Total Length of Fwd Packets  \
0                              0                           12   
1                              1                            6   
2                              1                            6   
3                              1                            6   
4                              0                           12   
...                          ...                          ...   
2830738                        2                          112   
2830739                        2                           84   
2830740                        1                           31   
2830741                        2                          192   
2830742                        2                          188   

          Total Length of Bwd Packets   Fwd Packet Length Max  \
0                                   0                       6   
1                                   6                       6   
2                                   6                       6   
3                                   6                       6   
4                                   0                       6   
...                               ...                     ...   
2830738                           152                      28   
2830739                           362                      42   
2830740                             6                      31   
2830741                           256                      32   
2830742                           226                      47   

          Fwd Packet Length Min   Fwd Packet Length Mean  \
0                             6                      6.0   
1                             6                      6.0   
2                             6                      6.0   
3                             6                      6.0   
4                             6                      6.0   
...                         ...                      ...   
2830738                      28                     28.0   
2830739                      42                     42.0   
2830740                       0                     15.5   
2830741                      32                     32.0   
2830742                      47                     47.0   

          Fwd Packet Length Std  ...   min_seg_size_forward  Active Mean  \
0                       0.00000  ...                     20          0.0   
1                       0.00000  ...                     20          0.0   
2                       0.00000  ...                     20          0.0   
3                       0.00000  ...                     20          0.0   
4                       0.00000  ...                     20          0.0   
...                         ...  ...                    ...          ...   
2830738                 0.00000  ...                     20          0.0   
2830739                 0.00000  ...                     20          0.0   
2830740                21.92031  ...                     32          0.0   
2830741                 0.00000  ...                     20          0.0   
2830742                 0.00000  ...                     2

In [ ]:
data.columns.tolist()

[' Destination Port',
 ' Flow Duration',
 ' Total Fwd Packets',
 ' Total Backward Packets',
 'Total Length of Fwd Packets',
 ' Total Length of Bwd Packets',
 ' Fwd Packet Length Max',
 ' Fwd Packet Length Min',
 ' Fwd Packet Length Mean',
 ' Fwd Packet Length Std',
 'Bwd Packet Length Max',
 ' Bwd Packet Length Min',
 ' Bwd Packet Length Mean',
 ' Bwd Packet Length Std',
 'Flow Bytes/s',
 ' Flow Packets/s',
 ' Flow IAT Mean',
 ' Flow IAT Std',
 ' Flow IAT Max',
 ' Flow IAT Min',
 'Fwd IAT Total',
 ' Fwd IAT Mean',
 ' Fwd IAT Std',
 ' Fwd IAT Max',
 ' Fwd IAT Min',
 'Bwd IAT Total',
 ' Bwd IAT Mean',
 ' Bwd IAT Std',
 ' Bwd IAT Max',
 ' Bwd IAT Min',
 'Fwd PSH Flags',
 ' Bwd PSH Flags',
 ' Fwd URG Flags',
 ' Bwd URG Flags',
 ' Fwd Header Length',
 ' Bwd Header Length',
 'Fwd Packets/s',
 ' Bwd Packets/s',
 ' Min Packet Length',
 ' Max Packet Length',
 ' Packet Length Mean',
 ' Packet Length Std',
 ' Packet Length Variance',
 'FIN Flag Count',
 ' SYN Flag Count',
 ' RST Flag Count',
 ' P

In [ ]:
data.columns[-1]

' Label'

In [ ]:
data[' Label'].value_counts()

,count
Label,
BENIGN,2273097
DoS Hulk,231073
PortScan,158930
DDoS,128027
DoS GoldenEye,10293
FTP-Patator,7938
SSH-Patator,5897
DoS slowloris,5796
DoS Slowhttptest,5499


In [ ]:
group_map = {
    'PortScan': [
        "netAccess(machine2,tcp,22)", "netAccess(machine3,tcp,22)",
        "netAccess(machine3,tcp,53)", "netAccess(machine1,tcp,445)"
    ],
    'SSH-Patator': [
        "logInService(machine3,tcp,22)", "logInService(machine2,tcp,22)",
        "principalCompromised(victim)"
    ],
    'FTP-Patator': [
        "canAccessHost(machine1)", "canAccessHost(machine2)", "canAccessHost(machine3)"
    ],
    'DDoS': [
        "execCode(machine3,root)", "execCode(machine3,user)"
    ],
    'DoS GoldenEye': [
        "execCode(machine1,root)", "execCode(machine1,user)"
    ],
    'DoS Slowhttptest': [
        "execCode(machine2,user)", "execCode(machine2,root)"
    ]
}

group_to_keyword = {g: g for g in group_map}

desc_to_group = {}
for group_name, desc_list in group_map.items():
    for desc in desc_list:
        desc_to_group[desc] = group_name

node_desc_map = {
    i: f"{node_dict[i]['predicate']}({','.join(node_dict[i]['attributes'])})"
    for i in node_dict
}

attack_keywords = []
for idx in action_node_idx:
    desc = node_desc_map[idx]
    group = desc_to_group[desc]
    attack_keywords.append(group_to_keyword[group])

print(attack_keywords)
print(len(attack_keywords))

from collections import Counter
print(Counter(attack_keywords))

['FTP-Patator', 'DDoS', 'SSH-Patator', 'DoS GoldenEye', 'PortScan', 'DoS Slowhttptest', 'DoS Slowhttptest', 'FTP-Patator', 'PortScan', 'DoS GoldenEye', 'FTP-Patator', 'SSH-Patator', 'PortScan', 'SSH-Patator']
14
Counter({'FTP-Patator': 3, 'SSH-Patator': 3, 'PortScan': 3, 'DoS GoldenEye': 2, 'DoS Slowhttptest': 2, 'DDoS': 1})


In [ ]:
data = data.drop(columns=[' Destination Port'], errors='ignore')
assert ' Destination Port' not in data.columns
print(data.shape)

(2830743, 78)


In [ ]:
x = data.iloc[:, :-1].values
y = data[' Label'].values

In [ ]:
x.shape

(2830743, 77)

In [ ]:
y.shape

(2830743,)

In [ ]:
x=np.where(np.isinf(x),np.nan,x)
mask=~np.isnan(x).any(axis=1)
x=x[mask]
y=y[mask]

perm = np.random.RandomState(42).permutation(len(x))
x = x[perm]
y = y[perm]
print("shuffled:", x.shape)

shuffled: (2827876, 77)


In [ ]:
x.shape

(2827876, 77)

In [ ]:
y.shape

(2827876,)

In [ ]:
num_action_nodes=len(action_node_idx)
num_benign=2400
num_malic=900

In [ ]:
total_benign=num_benign*num_action_nodes+num_malic*num_action_nodes*num_action_nodes

In [ ]:
total_benign

210000

In [ ]:
benign_mask=(y=="BENIGN")
benign_rows=x[benign_mask]
benign_rows.shape[0]

2271320

In [ ]:
benign_rows=benign_rows[:total_benign]
benign=benign_rows.reshape(-1,num_action_nodes,x.shape[1])

In [ ]:
benign.shape

(15000, 14, 77)

In [ ]:
x_benign=benign[:num_benign].copy()
y_benign=np.zeros((x_benign.shape[0],x_benign.shape[1]))

In [ ]:
x_benign.shape

(2400, 14, 77)

In [ ]:
y_benign.shape

(2400, 14)

In [ ]:
x_malic=benign[num_benign:num_benign+num_malic*num_action_nodes].copy()
y_malic=np.zeros((x_malic.shape[0],x_malic.shape[1]))

In [ ]:
x_malic.shape

(12600, 14, 77)

In [ ]:
y_malic.shape

(12600, 14)

In [ ]:
used_indices = {}

for idx in range(num_action_nodes):
    keyword = attack_keywords[idx]
    attack_mask = np.array([keyword in label for label in y])
    attack_rows = x[attack_mask]

    start_offset = used_indices.get(keyword, 0)
    end_offset = start_offset + num_malic

    print(f"Node {idx}: keyword='{keyword}', total available={attack_rows.shape[0]}, "
          f"taking rows[{start_offset}:{end_offset}]")

    if end_offset > attack_rows.shape[0]:
        print(f"  WARNING: sirf {attack_rows.shape[0]} rows hain, {end_offset} chahiye — SKIP karna padega")
        continue

    x_malic[idx*num_malic:(idx+1)*num_malic, idx, :] = attack_rows[start_offset:end_offset]
    used_indices[keyword] = end_offset
    y_malic[idx*num_malic:(idx+1)*num_malic, idx] = 1

print("\nDone. Final used_indices per keyword:", used_indices)

Node 0: keyword='FTP-Patator', total available=7935, taking rows[0:900]
Node 1: keyword='DDoS', total available=128025, taking rows[0:900]
Node 2: keyword='SSH-Patator', total available=5897, taking rows[0:900]
Node 3: keyword='DoS GoldenEye', total available=10293, taking rows[0:900]
Node 4: keyword='PortScan', total available=158804, taking rows[0:900]
Node 5: keyword='DoS Slowhttptest', total available=5499, taking rows[0:900]
Node 6: keyword='DoS Slowhttptest', total available=5499, taking rows[900:1800]
Node 7: keyword='FTP-Patator', total available=7935, taking rows[900:1800]
Node 8: keyword='PortScan', total available=158804, taking rows[900:1800]
Node 9: keyword='DoS GoldenEye', total available=10293, taking rows[900:1800]
Node 10: keyword='FTP-Patator', total available=7935, taking rows[1800:2700]
Node 11: keyword='SSH-Patator', total available=5897, taking rows[900:1800]
Node 12: keyword='PortScan', total available=158804, taking rows[1800:2700]
Node 13: keyword='SSH-Patator'

In [ ]:
x_malic.shape

(12600, 14, 77)

In [ ]:
y_malic.shape

(12600, 14)

In [ ]:
y_malic.sum()

np.float64(12600.0)

In [ ]:
y_malic

array([[1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(12600, 14))

In [ ]:
num_nodes=len(nodes)

In [ ]:
rt_meas_dim=x_benign.shape[2]

In [ ]:
rt_meas_dim

77

In [ ]:
X_benign=np.zeros((num_benign,num_nodes,rt_meas_dim))
X_benign[:, action_node_idx, :]=x_benign
Y_benign=y_benign

In [ ]:
X_malic=np.zeros((num_malic*num_action_nodes,num_nodes,rt_meas_dim))
X_malic[:,action_node_idx,:]=x_malic
Y_malic=y_malic

In [ ]:
X=np.concatenate((X_benign,X_malic),axis=0)
Y=np.concatenate((Y_benign,Y_malic),axis=0)

In [ ]:
X.shape

(15000, 51, 77)

In [ ]:
Y.shape

(15000, 14)

In [ ]:
print(num_action_nodes)
print(X.shape)
print(Y.shape)
print(y_malic.sum())

14
(15000, 51, 77)
(15000, 14)
12600.0


### Node Features

In [ ]:
num_samples=X.shape[0]

In [ ]:
node_feat_ts = np.stack([node_features for _ in range(num_samples)], axis=0)

In [ ]:
node_feat_ts.shape

(15000, 51, 73)

In [ ]:
X=np.concatenate((node_feat_ts,X),axis=2)

In [ ]:
X.shape

(15000, 51, 150)

In [ ]:
Y.shape

(15000, 14)

### Dataset Split

In [ ]:
X_t=torch.tensor(X,dtype=torch.float32)
Y_t=torch.tensor(Y,dtype=torch.float32)
edge_index_t=torch.tensor(edge_index,dtype=torch.long)

In [ ]:
X_train,X_test,Y_train,Y_test=train_test_split(X_t,Y_t,test_size=0.2,random_state=42,stratify=Y_t)

In [ ]:
X_train,X_val,Y_train,Y_val=train_test_split(X_train,Y_train,test_size=0.25,random_state=42,stratify=Y_train)

### safe scaling 

In [ ]:
from sklearn.metrics import f1_score as _f1, precision_score as _prec, recall_score as _rec
node_feat_dim = node_features.shape[1]

scaler = MinMaxScaler(feature_range=(0, 1))
_tr = X_train[:, action_node_idx, node_feat_dim:]
scaler.fit(_tr.reshape(-1, _tr.shape[-1]).numpy())

def apply_scaler(X):
    X = X.clone()
    sub = X[:, action_node_idx, node_feat_dim:]
    shp = sub.shape
    X[:, action_node_idx, node_feat_dim:] = torch.tensor(
        scaler.transform(sub.reshape(-1, shp[-1]).numpy()), dtype=torch.float32).reshape(shp)
    return X

X_train, X_val, X_test = apply_scaler(X_train), apply_scaler(X_val), apply_scaler(X_test)
print("scaler fit on", _tr.shape[0] * _tr.shape[1], "TRAIN traffic rows; applied to train / val / test")

scaler fit on 126000 TRAIN traffic rows; applied to train / val / test


In [ ]:
X_train.shape

torch.Size([9000, 51, 150])

In [ ]:
Y_train.shape

torch.Size([9000, 14])

In [ ]:
X_val.shape

torch.Size([3000, 51, 150])

In [ ]:
Y_val.shape

torch.Size([3000, 14])

In [ ]:
X_test.shape

torch.Size([3000, 51, 150])

In [ ]:
Y_test.shape

torch.Size([3000, 14])

In [ ]:
edge_index_t.shape

torch.Size([2, 72])

In [ ]:
print(action_node_idx)
print(len(node_dict))

[0, 2, 5, 7, 9, 17, 21, 23, 27, 31, 33, 36, 44, 48]
51


### NN Model — Multi-Seed Training

In [ ]:
rt_meas_dim = 77
X_train_nn = X_train[:, action_node_idx, -rt_meas_dim:]
X_val_nn = X_val[:, action_node_idx, -rt_meas_dim:]
X_test_nn = X_test[:, action_node_idx, -rt_meas_dim:]

in_dim_nn = rt_meas_dim
hidden_dim = 20
out_dim = 1

num_epochs_nn = 150
seeds = [42, 123, 999, 7, 2024]

In [ ]:
def train_and_evaluate_nn(seed):

    torch.manual_seed(seed)

    model_nn = NN(in_dim_nn, hidden_dim, out_dim).to(device)

    num_class_0_nn = (Y_train == 0).sum().item()
    num_class_1_nn = (Y_train == 1).sum().item()
    pos_weight_nn = torch.tensor([num_class_0_nn / num_class_1_nn], dtype=torch.float32).to(device)
    criterion_nn = nn.BCEWithLogitsLoss(pos_weight=pos_weight_nn)

    optimizer_nn = optim.Adam(model_nn.parameters(), lr=0.001)

    def evaluate(X_data, Y_data):
        model_nn.eval()
        with torch.no_grad():
            X_data = X_data.to(device)
            Y_data = Y_data.to(device)
            output = model_nn(X_data)
            loss = criterion_nn(output, Y_data)
            pred = (torch.sigmoid(output) > 0.5).float()
            accuracy = (pred == Y_data).sum().item() / Y_data.numel()
        return loss.item(), accuracy

    best_val_loss = float('inf')
    best_state = None

    train_dataset = TensorDataset(X_train_nn, Y_train)
    train_loader = DataLoader(train_dataset, batch_size=30, shuffle=False)

    for epoch in range(num_epochs_nn):
        model_nn.train()

        for batch_X, batch_Y in train_loader:
            batch_X = batch_X.to(device)
            batch_Y = batch_Y.to(device)

            output = model_nn(batch_X)
            loss = criterion_nn(output, batch_Y)

            optimizer_nn.zero_grad()
            loss.backward()
            optimizer_nn.step()

        val_loss, val_acc = evaluate(X_val_nn, Y_val)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model_nn.state_dict().items()}

    model_nn.load_state_dict(best_state)
    model_nn.eval()

    torch.save(model_nn.state_dict(), f"best_nn_fixed_seed{seed}.pt")

    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for i in range(len(X_test_nn)):
            x = X_test_nn[i].to(device)
            y = Y_test[i].to(device)

            output = model_nn(x)
            prob = torch.sigmoid(output)
            pred = (prob > 0.5).float()

            y_true.extend(y.cpu().numpy().reshape(-1))
            y_pred.extend(pred.cpu().numpy().reshape(-1))
            y_prob.extend(prob.cpu().numpy().reshape(-1))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        'seed': seed,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'auc': roc_auc_score(y_true, y_prob),
        'fpr': fp / (fp + tn),
        'fnr': fn / (fn + tp),
        'precision_macro': _prec(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': _rec(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro': _f1(y_true, y_pred, average='macro', zero_division=0),
    }

In [ ]:
results = []

for seed in seeds:
    print(f"--- Training with seed {seed} ---")
    metrics = train_and_evaluate_nn(seed)
    results.append(metrics)
    print(metrics)
    print()

--- Training with seed 42 ---
{'seed': 42, 'accuracy': 0.946, 'precision': 0.5264817150063051, 'recall': 0.9940476190476191, 'f1': 0.6883759274525969, 'auc': np.float64(0.9945321118991332), 'fpr': np.float64(0.05706686930091186), 'fnr': np.float64(0.005952380952380952), 'precision_macro': 0.7630394719706892, 'recall_macro': 0.9684903748733535, 'f1_macro': 0.8294073271422677}

--- Training with seed 123 ---
{'seed': 123, 'accuracy': 0.9579047619047619, 'precision': 0.5884705882352941, 'recall': 0.9924603174603175, 'f1': 0.7388478581979321, 'auc': np.float64(0.9952287626043326), 'fpr': np.float64(0.04430091185410334), 'fnr': np.float64(0.00753968253968254), 'precision_macro': 0.793983638488508, 'recall_macro': 0.9740797028031071, 'f1_macro': 0.8579775999522614}

--- Training with seed 999 ---
{'seed': 999, 'accuracy': 0.9602857142857143, 'precision': 0.6023546371936569, 'recall': 0.9948412698412699, 'f1': 0.7503741394791978, 'auc': np.float64(0.9957534506119232), 'fpr': np.float64(0.0419

### Multi-Seed Summary

In [ ]:
results_df = pd.DataFrame(results)
results_df

,seed,accuracy,precision,recall,f1,auc,fpr,fnr,precision_macro,recall_macro,f1_macro
0,42,0.946000,0.526482,0.994048,0.688376,0.994532,0.057067,0.005952,0.763039,0.968490,0.829407
1,123,0.957905,0.588471,0.992460,0.738848,0.995229,0.044301,0.007540,0.793984,0.974080,0.857978
2,999,0.960286,0.602355,0.994841,0.750374,0.995753,0.041920,0.005159,0.801006,0.976461,0.864400
3,7,0.960310,0.602845,0.992063,0.749963,0.995203,0.041717,0.007937,0.801159,0.975173,0.864203
4,2024,0.950571,0.548684,0.992857,0.706780,0.994596,0.052128,0.007143,0.774102,0.970365,0.839895


In [ ]:
print("--- Mean ± Std across", len(seeds), "seeds ---")
for col in ['accuracy', 'precision', 'recall', 'f1', 'auc', 'fpr', 'fnr']:
    print(f"{col:10s}: {results_df[col].mean():.4f} ± {results_df[col].std():.4f}")

--- Mean ± Std across 5 seeds ---
accuracy  : 0.9550 ± 0.0064
precision : 0.5738 ± 0.0344
recall    : 0.9933 ± 0.0012
f1        : 0.7269 ± 0.0279
auc       : 0.9951 ± 0.0005
fpr       : 0.0474 ± 0.0069
fnr       : 0.0067 ± 0.0012


### Summary attack-class metrics + macro for reference, saved to CSV

In [ ]:
results_df = pd.DataFrame(results)
cols = ['accuracy', 'precision', 'recall', 'f1', 'auc', 'fpr', 'fnr', 'precision_macro', 'recall_macro', 'f1_macro']
print("--- NN (FIXED) | mean +- std across", len(seeds), "seeds | precision/recall/f1 = attack class ---")
for col in cols:
    print(f"{col:16s}: {results_df[col].mean():.4f} +- {results_df[col].std():.4f}")
results_df.to_csv("results_nn_fixed.csv", index=False)
print("\nSaved: results_nn_fixed.csv")

--- NN (FIXED) | mean +- std across 5 seeds | precision/recall/f1 = attack class ---
accuracy        : 0.9550 +- 0.0064
precision       : 0.5738 +- 0.0344
recall          : 0.9933 +- 0.0012
f1              : 0.7269 +- 0.0279
auc             : 0.9951 +- 0.0005
fpr             : 0.0474 +- 0.0069
fnr             : 0.0067 +- 0.0012
precision_macro : 0.7867 +- 0.0172
recall_macro    : 0.9729 +- 0.0034
f1_macro        : 0.8512 +- 0.0157

Saved: results_nn_fixed.csv
